In [1]:
# ============================================
# 1. Imports
# ============================================
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split, StratifiedKFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# For reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [2]:
#connect to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#load data
data = np.load("/content/drive/MyDrive/AML/AML_resnet_features.npz",allow_pickle=True)
clinical_df=pd.read_csv('/content/drive/MyDrive/AML/cleaned_clinical_data.csv')
print(data.files)

['X_img', 'y_img', 'pids_img']


In [4]:
#Inspect raw arrays
X_img = data["X_img"]
y_img = data["y_img"]
pids_img = data["pids_img"]

print("Image feature shape:", X_img.shape)
print("Image labels shape:", y_img.shape)
print("Patient IDs shape:", pids_img.shape)
print("Unique labels in NPZ:", np.unique(y_img))

Image feature shape: (81220, 2048)
Image labels shape: (81220,)
Patient IDs shape: (81220,)
Unique labels in NPZ: [0 1 2 3 4]


In [5]:
df = pd.DataFrame(data["X_img"])
df["patient_id"] = data["pids_img"]
df["label"] = data["y_img"]
df.head()

,0,1,2,3,4,5,6,7,8,9,...,2040,2041,2042,2043,2044,2045,2046,2047,patient_id,label
0,2.572266,0.236158,0.001865,0.012011,0.577528,0.007699,0.592216,0.020741,0.204100,0.000000,...,0.003160,1.940869,0.026732,0.336324,0.000000,0.251699,0.761808,0.744895,FOH,1
1,3.140793,0.227892,0.000000,0.000210,0.009509,0.000000,0.103997,0.066837,0.101553,0.000000,...,0.284081,1.615394,0.009393,0.069238,0.133527,0.146256,0.318404,0.370203,FOH,1
2,3.353043,0.189777,0.000000,0.060990,0.061442,0.000000,1.003086,0.007684,0.000000,0.000000,...,0.017340,1.297336,0.000000,0.014775,0.006497,0.578233,2.318575,1.131269,FOH,1
3,1.777953,0.322329,0.000000,0.000000,0.406503,0.000000,0.176038,0.002690,0.044931,0.000000,...,0.198050,0.524655,0.010835,0.307512,0.392992,0.228123,0.556874,0.243334,FOH,1
4,1.083054,0.676473,0.010175,0.000000,0.032757,0.085341,0.245873,0.195111,0.029267,0.028444,...,0.157330,0.530932,0.184369,0.077602,0.282538,0.419215,0.021608,0.288568,FOH,1


In [6]:
#Aggregate image features to patient level

def aggregate_patient_features(X, patient_ids, labels):
    df_feat = pd.DataFrame(X)
    df_feat["patient_id"] = patient_ids
    df_feat["target"] = labels

    feature_cols = df_feat.columns[:-2]

    grouped_X = df_feat.groupby("patient_id")[feature_cols].mean()
    grouped_y = df_feat.groupby("patient_id")["target"].first()

    X_out = grouped_X.values
    y_out = grouped_y.values.astype(int)
    pids_out = grouped_X.index.values

    return X_out, y_out, pids_out

X_img_patient, y_patient, patient_ids_unique = aggregate_patient_features(
    X_img, pids_img, y_img
)

print("Patient-level image feature shape:", X_img_patient.shape)
print("Patient-level labels shape:", y_patient.shape)
print("Unique patients:", len(patient_ids_unique))
print("Patient-level class distribution:", np.bincount(y_patient))

Patient-level image feature shape: (189, 2048)
Patient-level labels shape: (189,)
Unique patients: 189
Patient-level class distribution: [60 36 24 32 37]


In [7]:
#Check label consistency per patient
df_check = pd.DataFrame(X_img)
df_check["patient_id"] = pids_img
df_check["target"] = y_img

label_consistency = df_check.groupby("patient_id")["target"].nunique()
print("Patients with inconsistent labels:", (label_consistency > 1).sum())

Patients with inconsistent labels: 0


In [8]:
#Fix clinical column names
print(clinical_df.columns)

Index(['patient_id', 'sex_1f_2m', 'age', 'leucocytes_per_ul', 'pb_myeloblast',
       'pb_promyelocyte', 'pb_myelocyte', 'pb_metamyelocyte',
       'pb_neutrophil_band', 'pb_neutrophil_segmented', 'pb_eosinophil',
       'pb_basophil', 'pb_monocyte', 'pb_lymph_typ', 'pb_lymph_atyp_react',
       'pb_other', 'target_multi'],
      dtype='object')


In [9]:
#train/test split
train_ids, test_ids, y_train_patients, y_test_patients = train_test_split(
    patient_ids_unique,
    y_patient,
    test_size=0.2,
    random_state=42,
    stratify=y_patient
)

print("Train patients:", len(train_ids))
print("Test patients:", len(test_ids))
print("Train class distribution:", np.bincount(y_train_patients))
print("Test class distribution:", np.bincount(y_test_patients))

Train patients: 151
Test patients: 38
Train class distribution: [48 29 19 26 29]
Test class distribution: [12  7  5  6  8]


In [10]:
#Build patient-level image dataframe
img_patient_df = pd.DataFrame(X_img_patient)
img_patient_df["patient_id"] = patient_ids_unique
img_patient_df["target"] = y_patient

print(img_patient_df.head())

          0         1         2         3         4         5         6  \
0  1.705238  0.794756  0.051396  0.088291  0.279537  0.088349  0.103345   
1  1.368541  0.668797  0.020908  0.051473  0.376527  0.062844  0.094116   
2  1.537848  0.443945  0.020140  0.104213  0.481685  0.056840  0.191044   
3  1.433469  0.923722  0.038816  0.138829  0.314218  0.111636  0.100506   
4  2.692181  0.591474  0.057644  0.145914  0.280207  0.119006  0.288530   

          7         8         9  ...      2040      2041      2042      2043  \
0  0.065124  0.102372  0.006527  ...  0.152627  0.657768  0.094092  0.126387   
1  0.065969  0.163312  0.019071  ...  0.178649  0.291505  0.213840  0.054693   
2  0.063573  0.116885  0.015504  ...  0.112776  0.249994  0.195131  0.113578   
3  0.072002  0.111956  0.010859  ...  0.262798  0.712209  0.069076  0.115205   
4  0.088622  0.185711  0.010654  ...  0.095617  0.635916  0.163337  0.284088   

       2044      2045      2046      2047  patient_id  target  
0  0

In [11]:
#build multimodal features

def build_multimodal_features(train_ids_subset, val_ids_subset, n_components=0.95):
    # ----- IMAGE BRANCH -----
    train_img_patient = img_patient_df.set_index("patient_id").loc[train_ids_subset].reset_index()
    val_img_patient = img_patient_df.set_index("patient_id").loc[val_ids_subset].reset_index()

    y_train_fold = train_img_patient["target"].values.astype(int)
    y_val_fold = val_img_patient["target"].values.astype(int)

    X_train_img_fold = train_img_patient.drop(columns=["patient_id", "target"]).values
    X_val_img_fold = val_img_patient.drop(columns=["patient_id", "target"]).values

    #scaler
    scaler_img = StandardScaler()
    X_train_img_scaled = scaler_img.fit_transform(X_train_img_fold)
    X_val_img_scaled = scaler_img.transform(X_val_img_fold)

    #pca
    pca = PCA(n_components=n_components)
    X_train_img_pca = pca.fit_transform(X_train_img_scaled)
    X_val_img_pca = pca.transform(X_val_img_scaled)

    # ----- CLINICAL BRANCH -----
    train_clinical = clinical_df.set_index("patient_id").loc[train_ids_subset].reset_index()
    val_clinical = clinical_df.set_index("patient_id").loc[val_ids_subset].reset_index()

    # Verify patient order and labels match
    assert np.array_equal(train_clinical["patient_id"].values, train_ids_subset)
    assert np.array_equal(val_clinical["patient_id"].values, val_ids_subset)
    assert np.array_equal(train_clinical["target_multi"].values.astype(int), y_train_fold)
    assert np.array_equal(val_clinical["target_multi"].values.astype(int), y_val_fold)

    drop_cols = ["patient_id", "target_multi"]
    X_train_clinical_fold = train_clinical.drop(columns=drop_cols).values
    X_val_clinical_fold = val_clinical.drop(columns=drop_cols).values

    #scaler
    scaler_clinical = StandardScaler()
    X_train_clinical_fold = scaler_clinical.fit_transform(X_train_clinical_fold)
    X_val_clinical_fold = scaler_clinical.transform(X_val_clinical_fold)

    # ----- FUSION -----
    X_train_final_fold = np.concatenate([X_train_img_pca, X_train_clinical_fold], axis=1)
    X_val_final_fold = np.concatenate([X_val_img_pca, X_val_clinical_fold], axis=1)

    preprocess_objects = {
        "scaler_img": scaler_img,
        "pca": pca,
        "scaler_clinical": scaler_clinical
    }

    return X_train_final_fold, X_val_final_fold, y_train_fold, y_val_fold, preprocess_objects

In [12]:
#CV helper functions
def evaluate_random_forest_cv(patient_ids_train, y_train_labels, params=None, n_splits=5):
    if params is None:
        params = {
            "n_estimators": 200,
            "max_depth": 8,
            "random_state": 42
        }

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_results = []

    patient_ids_train = np.array(patient_ids_train)
    y_train_labels = np.array(y_train_labels)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(patient_ids_train, y_train_labels), 1):
        fold_train_ids = patient_ids_train[tr_idx]
        fold_val_ids = patient_ids_train[val_idx]

        X_tr, X_val, y_tr, y_val, _ = build_multimodal_features(fold_train_ids, fold_val_ids)

        model = RandomForestClassifier(**params)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        fold_results.append({
            "fold": fold,
            "accuracy": accuracy_score(y_val, y_pred),
            "macro_f1": f1_score(y_val, y_pred, average="macro"),
            "weighted_f1": f1_score(y_val, y_pred, average="weighted")
        })

    return pd.DataFrame(fold_results)

In [13]:
def evaluate_xgboost_cv(patient_ids_train, y_train_labels, params=None, n_splits=5):
    if params is None:
        params = {
            "objective": "multi:softprob",
            "num_class": len(np.unique(y_train_labels)),
            "n_estimators": 200,
            "max_depth": 4,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "random_state": 42,
            "eval_metric": "mlogloss"
        }

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_results = []

    patient_ids_train = np.array(patient_ids_train)
    y_train_labels = np.array(y_train_labels)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(patient_ids_train, y_train_labels), 1):
        fold_train_ids = patient_ids_train[tr_idx]
        fold_val_ids = patient_ids_train[val_idx]

        X_tr, X_val, y_tr, y_val, _ = build_multimodal_features(fold_train_ids, fold_val_ids)

        sample_weights = compute_sample_weight(class_weight="balanced", y=y_tr)

        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr, sample_weight=sample_weights)
        y_pred = model.predict(X_val).astype(int)

        fold_results.append({
            "fold": fold,
            "accuracy": accuracy_score(y_val, y_pred),
            "macro_f1": f1_score(y_val, y_pred, average="macro"),
            "weighted_f1": f1_score(y_val, y_pred, average="weighted")
        })

    return pd.DataFrame(fold_results)

In [14]:
def build_mlp(input_dim, num_classes, params=None):
    if params is None:
        params = {
            "dense1": 64,
            "dropout1": 0.4,
            "dense2": 32,
            "dropout2": 0.3,
            "learning_rate": 1e-3
        }

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(params["dense1"], activation="relu"),
        tf.keras.layers.Dropout(params["dropout1"]),
        tf.keras.layers.Dense(params["dense2"], activation="relu"),
        tf.keras.layers.Dropout(params["dropout2"]),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=params["learning_rate"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [15]:
def evaluate_mlp_cv(patient_ids_train, y_train_labels, params=None, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_results = []

    patient_ids_train = np.array(patient_ids_train)
    y_train_labels = np.array(y_train_labels)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(patient_ids_train, y_train_labels), 1):
        print(f"Training MLP fold {fold}...")

        fold_train_ids = patient_ids_train[tr_idx]
        fold_val_ids = patient_ids_train[val_idx]

        X_tr, X_val, y_tr, y_val, _ = build_multimodal_features(fold_train_ids, fold_val_ids)

        tf.keras.backend.clear_session()
        tf.random.set_seed(42)

        classes = np.unique(y_tr)
        class_weights_values = compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=y_tr
        )
        class_weights = dict(zip(classes, class_weights_values))

        model = build_mlp(
            input_dim=X_tr.shape[1],
            num_classes=len(np.unique(y_train_labels)),
            params=params
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=50,
            batch_size=16,
            class_weight=class_weights,
            callbacks=[early_stop],
            verbose=0
        )

        y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)

        fold_results.append({
            "fold": fold,
            "accuracy": accuracy_score(y_val, y_pred),
            "macro_f1": f1_score(y_val, y_pred, average="macro"),
            "weighted_f1": f1_score(y_val, y_pred, average="weighted")
        })

    return pd.DataFrame(fold_results)

In [16]:
# Run CV for all 3 models on TRAINING SET only
rf_cv = evaluate_random_forest_cv(train_ids, y_train_patients)
xgb_cv = evaluate_xgboost_cv(train_ids, y_train_patients)
mlp_cv = evaluate_mlp_cv(train_ids, y_train_patients)

print("RF CV results:")
print(rf_cv)
print()

print("XGB CV results:")
print(xgb_cv)
print()

print("MLP CV results:")
print(mlp_cv)

Training MLP fold 1...
Training MLP fold 2...
Training MLP fold 3...
Training MLP fold 4...
Training MLP fold 5...


RF CV results:
   fold  accuracy  macro_f1  weighted_f1
0     1  0.709677  0.640839     0.703542
1     2  0.766667  0.615931     0.718672
2     3  0.600000  0.521538     0.604615
3     4  0.766667  0.737363     0.763553
4     5  0.700000  0.638974     0.690085

XGB CV results:
   fold  accuracy  macro_f1  weighted_f1
0     1  0.741935  0.707026     0.740636
1     2  0.833333  0.804755     0.833846
2     3  0.766667  0.714286     0.765079
3     4  0.833333  0.798095     0.824286
4     5  0.666667  0.621299     0.662251

MLP CV results:
   fold  accuracy  macro_f1  weighted_f1
0     1  0.645161  0.581616     0.651483
1     2  0.666667  0.591966     0.667521
2     3  0.700000  0.595455     0.678355
3     4  0.733333  0.691898     0.735120
4     5  0.766667  0.763452     0.778204


In [17]:
#Summarize CV performance

cv_summary = pd.DataFrame([
    {
        "Model": "Random Forest",
        "CV Accuracy Mean": rf_cv["accuracy"].mean(),
        "CV Accuracy Std": rf_cv["accuracy"].std(),
        "CV Macro F1 Mean": rf_cv["macro_f1"].mean(),
        "CV Macro F1 Std": rf_cv["macro_f1"].std(),
        "CV Weighted F1 Mean": rf_cv["weighted_f1"].mean(),
        "CV Weighted F1 Std": rf_cv["weighted_f1"].std()
    },
    {
        "Model": "XGBoost",
        "CV Accuracy Mean": xgb_cv["accuracy"].mean(),
        "CV Accuracy Std": xgb_cv["accuracy"].std(),
        "CV Macro F1 Mean": xgb_cv["macro_f1"].mean(),
        "CV Macro F1 Std": xgb_cv["macro_f1"].std(),
        "CV Weighted F1 Mean": xgb_cv["weighted_f1"].mean(),
        "CV Weighted F1 Std": xgb_cv["weighted_f1"].std()
    },
    {
        "Model": "MLP",
        "CV Accuracy Mean": mlp_cv["accuracy"].mean(),
        "CV Accuracy Std": mlp_cv["accuracy"].std(),
        "CV Macro F1 Mean": mlp_cv["macro_f1"].mean(),
        "CV Macro F1 Std": mlp_cv["macro_f1"].std(),
        "CV Weighted F1 Mean": mlp_cv["weighted_f1"].mean(),
        "CV Weighted F1 Std": mlp_cv["weighted_f1"].std()
    }
]).sort_values("CV Weighted F1 Mean", ascending=False)

print("\n" + "="*70)
print("Cross-Validation Comparison")
print("="*70)
print(cv_summary)


Cross-Validation Comparison
           Model  CV Accuracy Mean  CV Accuracy Std  CV Macro F1 Mean  \
1        XGBoost          0.768387         0.069796          0.729092   
2            MLP          0.702366         0.049107          0.644877   
0  Random Forest          0.708602         0.068214          0.630929   

   CV Macro F1 Std  CV Weighted F1 Mean  CV Weighted F1 Std  
1         0.075518             0.765220            0.069651  
2         0.079867             0.702137            0.052924  
0         0.076945             0.696094            0.058139  


In [18]:
#Choose best model by CV Weighted F1 Mean
best_model_name = cv_summary.iloc[0]["Model"]
print("Best model from CV:", best_model_name)

Best model from CV: XGBoost


In [19]:
# Hyperparameter tuning for XGBoost only

from sklearn.model_selection import ParameterSampler

def tune_xgboost(patient_ids_train, y_train_labels, n_iter=20):
    param_grid = {
        "n_estimators": [100, 200, 300, 500],
        "max_depth": [3, 4, 5, 6, 8],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "subsample": [0.6, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "min_child_weight": [1, 3, 5],
        "gamma": [0, 0.1, 0.3, 0.5]
    }

    sampled_params = list(ParameterSampler(param_grid, n_iter=n_iter, random_state=42))

    best_score = -1
    best_params = None
    best_fold_scores = None
    all_trials = []

    for i, params in enumerate(sampled_params, 1):
        full_params = {
            **params,
            "objective": "multi:softprob",
            "num_class": len(np.unique(y_train_labels)),
            "random_state": 42,
            "eval_metric": "mlogloss"
        }

        print(f"\nTuning XGBoost {i}/{len(sampled_params)}")
        print(full_params)

        fold_df = evaluate_xgboost_cv(
            patient_ids_train,
            y_train_labels,
            params=full_params,
            n_splits=5
        )

        mean_acc = fold_df["accuracy"].mean()
        mean_macro = fold_df["macro_f1"].mean()
        mean_weighted = fold_df["weighted_f1"].mean()

        all_trials.append({
            "trial": i,
            **params,
            "cv_accuracy_mean": mean_acc,
            "cv_macro_f1_mean": mean_macro,
            "cv_weighted_f1_mean": mean_weighted
        })

        print("Mean Accuracy:", mean_acc)
        print("Mean Macro F1:", mean_macro)
        print("Mean Weighted F1:", mean_weighted)

        if mean_weighted > best_score:
            best_score = mean_weighted
            best_params = full_params
            best_fold_scores = fold_df.copy()

    trials_df = pd.DataFrame(all_trials).sort_values(
        "cv_weighted_f1_mean", ascending=False
    ).reset_index(drop=True)

    return best_params, best_score, best_fold_scores, trials_df

In [20]:
best_xgb_params, best_xgb_cv_score, best_xgb_folds, xgb_trials_df = tune_xgboost(
    train_ids,
    y_train_patients,
    n_iter=20
)

print("\n" + "="*70)
print("Best Tuned XGBoost Parameters")
print("="*70)
print(best_xgb_params)

print("\nBest CV Weighted F1:", best_xgb_cv_score)

print("\nBest fold-wise results:")
print(best_xgb_folds)

print("\nTop tuning trials:")
print(xgb_trials_df.head(10))


Tuning XGBoost 1/20
{'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.01, 'gamma': 0.3, 'colsample_bytree': 1.0, 'objective': 'multi:softprob', 'num_class': 5, 'random_state': 42, 'eval_metric': 'mlogloss'}
Mean Accuracy: 0.7219354838709677
Mean Macro F1: 0.666972360972361
Mean Weighted F1: 0.7151441628860983

Tuning XGBoost 2/20
{'subsample': 1.0, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.01, 'gamma': 0.1, 'colsample_bytree': 0.6, 'objective': 'multi:softprob', 'num_class': 5, 'random_state': 42, 'eval_metric': 'mlogloss'}
Mean Accuracy: 0.7350537634408603
Mean Macro F1: 0.6793426573426572
Mean Weighted F1: 0.7268299263460554

Tuning XGBoost 3/20
{'subsample': 1.0, 'n_estimators': 100, 'min_child_weight': 5, 'max_depth': 8, 'learning_rate': 0.03, 'gamma': 0.5, 'colsample_bytree': 0.8, 'objective': 'multi:softprob', 'num_class': 5, 'random_state': 42, 'eval_metric': 'mlogloss'}
Mean Accuracy: 0.728387

In [21]:
# Build final train/test features
X_train_final, X_test_final, y_train, y_test, preprocess_objects = build_multimodal_features(
    train_ids,
    test_ids
)

print("Final train shape:", X_train_final.shape)
print("Final test shape:", X_test_final.shape)
print("Final train labels:", np.bincount(y_train))
print("Final test labels:", np.bincount(y_test))
print("PCA components selected:", preprocess_objects["pca"].n_components_)

Final train shape: (151, 59)
Final test shape: (38, 59)
Final train labels: [48 29 19 26 29]
Final test labels: [12  7  5  6  8]
PCA components selected: 44


In [22]:
# Train final tuned XGBoost
final_sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

final_xgb = XGBClassifier(**best_xgb_params)

final_xgb.fit(
    X_train_final,
    y_train,
    sample_weight=final_sample_weights
)

final_xgb_pred = final_xgb.predict(X_test_final).astype(int)

In [23]:
# Final evaluation
print("\n" + "="*70)
print("Final Test Results - Tuned XGBoost")
print("="*70)

print("Accuracy:", accuracy_score(y_test, final_xgb_pred))
print("Macro F1:", f1_score(y_test, final_xgb_pred, average="macro"))
print("Weighted F1:", f1_score(y_test, final_xgb_pred, average="weighted"))

print("\nClassification Report:")
print(classification_report(y_test, final_xgb_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, final_xgb_pred))


Final Test Results - Tuned XGBoost
Accuracy: 0.8421052631578947
Macro F1: 0.807158371040724
Weighted F1: 0.8388509168849726

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        12
           1       0.83      0.71      0.77         7
           2       1.00      0.60      0.75         5
           3       0.57      0.67      0.62         6
           4       0.89      1.00      0.94         8

    accuracy                           0.84        38
   macro avg       0.84      0.80      0.81        38
weighted avg       0.85      0.84      0.84        38

Confusion Matrix:
[[12  0  0  0  0]
 [ 0  5  0  2  0]
 [ 1  0  3  1  0]
 [ 0  1  0  4  1]
 [ 0  0  0  0  8]]
